# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library. Every entity such as record sets, fields, and columns is referenced by its Croissant `@id` for clarity and reproducibility.

### Dataset Source
The dataset metadata is accessible and can be loaded using the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library if it's not installed yet
!pip install mlcroissant --quiet

## 1. Data Loading

We begin by loading the Croissant metadata and retrieving high-level dataset properties using `mlcroissant`. This metadata allows us to inspect schema-level information before loading the actual records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata
print(f"Dataset Name: {metadata.name}\n")
print("Description:")
print(metadata.description)
print("\nPublished on:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview

Next, we explore the Croissant schema for available record sets (tables), their `@id`s, and fields. All referencing is performed using `@id`. Let’s enumerate each record set and its structure.

In [ ]:
# List all record sets with their @id and name
print("Record sets in this dataset:")
record_sets = []

# Croissant v1.0 record sets are usually in metadata.record_set and have @id
for record_set in getattr(metadata, 'record_set', []):
    # Each record set should have @id and name attributes
    rec_id = record_set['@id'] if isinstance(record_set, dict) else getattr(record_set, '@id', None)
    rec_name = record_set.get('name') if isinstance(record_set, dict) else getattr(record_set, 'name', None)
    print(f"- @id: {rec_id}, name: {rec_name}")
    record_sets.append(rec_id)

if not record_sets:
    print("No explicit record sets listed in metadata; attempting to enumerate via dataset.records().")
    # Try to parse the recordSet(s) literally from the live ds object
    try:
        # This will hit the schema directly, see what record_set IDs are available
        croissant_obj = ds._croissant
        record_sets_nodes = croissant_obj.get('recordSet', [])
        if isinstance(record_sets_nodes, dict):
            record_sets_nodes = [record_sets_nodes]
        for rs in record_sets_nodes:
            rec_id = rs.get('@id', None)
            rec_name = rs.get('name', None)
            print(f"- @id: {rec_id}, name: {rec_name}")
            record_sets.append(rec_id)
    except Exception as e:
        print("[Error] Could not extract record set IDs from schema.")
        record_sets = []

print(f"\nRecord set @id(s) found: {record_sets}")
if not record_sets:
    print("No record set found. Please check the Croissant schema for structure.")

# If at least one record set, show fields for the first one
if record_sets:
    print(f"\nFields (by @id and name) for record set: {record_sets[0]}")
    # Croissant 1.0 pattern; fields inside record_set['field'] or 'fields' or as Field objects
    croissant_obj = ds._croissant
    record_set_nodes = croissant_obj.get('recordSet', [])
    if isinstance(record_set_nodes, dict):
        record_set_nodes = [record_set_nodes]
    target_record_set = None
    for node in record_set_nodes:
        if node.get('@id', None) == record_sets[0]:
            target_record_set = node
            break
    if target_record_set:
        fields = target_record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field.get('@id', None)
            field_name = field.get('name', None)
            print(f"- @id: {field_id}, name: {field_name}")

## 3. Data Extraction

Let’s load data from one or more record sets into Pandas DataFrames for further analysis. **All record set and field references use their `@id`.**

In [ ]:
# Specify the record set(s) to analyze (fill with IDs from the overview step).
# For this dataset, you may need to look up and update the @id below explicitly if there is only one known.

if not record_sets:
    raise RuntimeError("No record_set @id discovered for data extraction.")

# Example: use the first record set found
record_sets_to_load = [record_sets[0]]

dataframes = {}

for rec_id in record_sets_to_load:
    # The mlcroissant 'record_set' argument expects the @id
    records = list(ds.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set @id: {rec_id}")
    print(f"Columns: {dataframes[rec_id].columns.tolist()}")
    display(dataframes[rec_id].head())

## 4. Exploratory Data Analysis (EDA)

Let’s process numeric fields: filter records by value, normalize a feature, and optionally group by another field. All references use strict `@id` notation. Please update field IDs below according to your exploration results above.

In [ ]:
# Pick the loaded DataFrame and define field @ids
record_set_id = record_sets_to_load[0]
df = dataframes[record_set_id]

# Example: Select a numeric field for filtering (e.g., "cr:MW" or similar from schema),
# and a possible grouping column. Update these with actual @id from the overview step above.
numeric_field = None
group_field = None

for col in df.columns:
    # Try to pick a likely numeric field and categorical field
    if ('mw' in col.lower() or 'weight' in col.lower()) and numeric_field is None:
        numeric_field = col  # prefer molecular weight
    elif ('abundance' in col.lower() or 'value' in col.lower()) and numeric_field is None:
        numeric_field = col
    elif ('type' in col.lower() or 'sample' in col.lower() or 'category' in col.lower()):
        group_field = col

if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]
if group_field is None:
    # Just pick the second column as example
    group_field = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"Numeric field selected for EDA: {numeric_field}")
print(f"Group field selected for EDA: {group_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

# Safe filtering
try:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, norm_col]].head())

    # Grouped stats
    if group_field in df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} after filtering, grouped by {group_field}:")
        display(grouped_df.head())

except Exception as e:
    print("Error during EDA step:", e)

## 5. Visualization

Explore numeric distributions and groupings with plots. Adjust field IDs as needed for insightful visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], kde=True, bins=30, color='teal')
plt.title(f"Distribution of field: {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If group field is categorical and there are a manageable number of categories
if group_field in df.columns and df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using its Croissant schema with `mlcroissant`, explored its structure by `@id`, and performed basic exploratory data analysis and visualization on a record set. All operations referenced schema fields by their Croissant `@id` for clarity and reproducibility.

- **Next steps:** You can extend this notebook by exploring additional record sets or fields, performing advanced feature engineering, or integrating with downstream ML workflows using the structured metadata for robust FAIR data principles.

*Dataset DOI*: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
